In [14]:
import os
import numpy as np
import pandas as pd
from google.colab import drive
from sklearn.linear_model import Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Tự động tìm file trên cả máy tính local lẫn Google Colab
train_path = 'restaurant_visits_train.csv' if os.path.exists('restaurant_visits_train.csv') else 'data/processed/restaurant_visits_train.csv'
test_path = 'restaurant_visits_test.csv' if os.path.exists('restaurant_visits_test.csv') else 'data/processed/restaurant_visits_test.csv'

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)
print("Số dòng tập Train:", train_df.shape[0])
print("Số dòng tập Test :", test_df.shape[0])

Số dòng tập Train: 8004
Số dòng tập Test : 2001


In [10]:
# 1. Ghép tạm 2 tập để One-Hot Encoding đồng bộ tất cả các cột
combined_df = pd.concat([train_df, test_df], axis=0, ignore_index=True)
df_ohe = pd.get_dummies(combined_df, columns=['air_genre_name', 'air_area_name'], drop_first=True, dtype=float)

# 2. Tách X (đặc trưng) và y_log (nhãn đã chuyển sang log1p)
X_ohe = df_ohe.drop(columns=['visitors'])
y_log = np.log1p(df_ohe['visitors'])

# 3. Cắt lại thành 2 tập Train và Test đúng số dòng ban đầu
train_len = len(train_df)
X_train_ohe = X_ohe.iloc[:train_len]
X_test_ohe = X_ohe.iloc[train_len:]
y_train_log = y_log.iloc[:train_len]
y_test_log = y_log.iloc[train_len:]

# 4. Huấn luyện mô hình Ridge (L2)
model_ridge = Ridge(alpha=10.0)
model_ridge.fit(X_train_ohe, y_train_log)

# 5. Dự đoán trên tập Test và quy đổi lại số khách thực tế
y_pred_log = model_ridge.predict(X_test_ohe)
y_pred_orig = np.maximum(0, np.expm1(y_pred_log)) # Nghịch đảo exp(x) - 1
y_test_orig = test_df['visitors']

# 6. Tính 3 độ đo RMSLE, RMSE, MAE
rmsle = np.sqrt(mean_squared_error(y_test_log, y_pred_log))
rmse = np.sqrt(mean_squared_error(y_test_orig, y_pred_orig))
mae = mean_absolute_error(y_test_orig, y_pred_orig)

print("=== 1. LINEAR REGRESSION (SKLEARN RIDGE) ===")
print(f"RMSLE : {rmsle:.4f}")
print(f"RMSE  : {rmse:.2f} khách")
print(f"MAE   : {mae:.2f} khách")

=== 1. LINEAR REGRESSION (SKLEARN RIDGE) ===
RMSLE : 0.7956
RMSE  : 18.43 khách
MAE   : 12.83 khách


In [11]:
class RidgeRegressionScratch:
    def __init__(self, alpha=10.0):
        self.alpha = alpha
        self.w = None

    def fit(self, X, y):
        # Thêm cột 1 vào đầu ma trận X để tính hệ số chặn bias w0
        X_b = np.hstack([np.ones((X.shape[0], 1)), X])

        # Tạo ma trận đơn vị I (bỏ qua vị trí [0,0] để không phạt hệ số w0)
        I = np.eye(X_b.shape[1])
        I[0, 0] = 0

        # Công thức nghiệm đóng: w = (X^T * X + alpha * I)^(-1) * X^T * y
        self.w = np.linalg.pinv(X_b.T @ X_b + self.alpha * I) @ X_b.T @ y

    def predict(self, X):
        X_b = np.hstack([np.ones((X.shape[0], 1)), X])
        return X_b @ self.w

# Tận dụng dữ liệu X_train_ohe và y_train_log từ Ô 2
lr_scratch = RidgeRegressionScratch(alpha=10.0)
lr_scratch.fit(X_train_ohe.values, y_train_log.values)

# Dự đoán và tính chỉ số
y_pred_log_scratch = lr_scratch.predict(X_test_ohe.values)
y_pred_orig_scratch = np.maximum(0, np.expm1(y_pred_log_scratch))

rmsle = np.sqrt(mean_squared_error(y_test_log, y_pred_log_scratch))
rmse = np.sqrt(mean_squared_error(y_test_orig, y_pred_orig_scratch))
mae = mean_absolute_error(y_test_orig, y_pred_orig_scratch)

print("=== 2. LINEAR REGRESSION (CODE CHAY RIDGE) ===")
print(f"RMSLE : {rmsle:.4f}")
print(f"RMSE  : {rmse:.2f} khách")
print(f"MAE   : {mae:.2f} khách")

=== 2. LINEAR REGRESSION (CODE CHAY RIDGE) ===
RMSLE : 0.7956
RMSE  : 18.43 khách
MAE   : 12.83 khách


In [12]:
# 1. Tạo bản sao dữ liệu và biến đổi log(1 + visitors)
train_cart = train_df.copy()
test_cart = test_df.copy()

train_cart['visitors_log'] = np.log1p(train_cart['visitors'])
test_cart['visitors_log'] = np.log1p(test_cart['visitors'])

# 2. Target Encoding: Tính trung bình log(visitors) theo từng danh mục trên tập Train
cat_cols = ['air_genre_name', 'air_area_name']
global_mean = train_cart['visitors_log'].mean()

for col in cat_cols:
    encoding_map = train_cart.groupby(col)['visitors_log'].mean().to_dict()
    train_cart[col] = train_cart[col].map(encoding_map).fillna(global_mean)
    test_cart[col] = test_cart[col].map(encoding_map).fillna(global_mean)

# 3. Tách biến đầu vào X và biến mục tiêu y
X_train_te = train_cart.drop(columns=['visitors', 'visitors_log'])
y_train_te = train_cart['visitors_log']

X_test_te = test_cart.drop(columns=['visitors', 'visitors_log'])
y_test_te = test_cart['visitors_log']

# 4. Huấn luyện cây CART với siêu tham số chống Overfit
cart_sk = DecisionTreeRegressor(
    max_depth=5,
    min_samples_split=20,
    min_samples_leaf=10,
    random_state=42
)
cart_sk.fit(X_train_te, y_train_te)

# 5. Dự đoán và quy đổi về số khách
y_pred_log_cart = cart_sk.predict(X_test_te)
y_pred_orig_cart = np.maximum(0, np.expm1(y_pred_log_cart))

# 6. Đánh giá kết quả
rmsle = np.sqrt(mean_squared_error(y_test_te, y_pred_log_cart))
rmse = np.sqrt(mean_squared_error(y_test_orig, y_pred_orig_cart))
mae = mean_absolute_error(y_test_orig, y_pred_orig_cart)

print("=== 3. CART REGRESSOR (SKLEARN) ===")
print(f"RMSLE : {rmsle:.4f}")
print(f"RMSE  : {rmse:.2f} khách")
print(f"MAE   : {mae:.2f} khách")

=== 3. CART REGRESSOR (SKLEARN) ===
RMSLE : 0.7950
RMSE  : 18.19 khách
MAE   : 12.79 khách


In [13]:
class DecisionTreeScratch:
    def __init__(self, max_depth=5, min_samples_split=20, min_samples_leaf=10):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.root = None

    # Tìm thuộc tính và ngưỡng cắt làm giảm Phương sai (Variance) nhiều nhất
    def _best_split(self, X, y):
        best_gain = -1
        best_feat, best_thresh = None, None
        n_samples, n_features = X.shape
        var_parent = np.var(y)

        for feat_idx in range(n_features):
            col = X[:, feat_idx]
            thresholds = np.unique(col)
            if len(thresholds) > 20:
                thresholds = np.percentile(col, np.linspace(0, 100, 20))

            for thresh in thresholds:
                left_mask = col <= thresh
                right_mask = ~left_mask

                n_left, n_right = np.sum(left_mask), np.sum(right_mask)

                # Dừng nếu số mẫu ở nút con ít hơn min_samples_leaf
                if n_left < self.min_samples_leaf or n_right < self.min_samples_leaf:
                    continue

                var_left = np.var(y[left_mask])
                var_right = np.var(y[right_mask])
                var_split = (n_left * var_left + n_right * var_right) / n_samples
                gain = var_parent - var_split

                if gain > best_gain:
                    best_gain = gain
                    best_feat = feat_idx
                    best_thresh = thresh

        return best_feat, best_thresh

    # Hàm xây dựng cây đệ quy đơn giản (lưu bằng dictionary)
    def _build_tree(self, X, y, depth=0):
        n_samples = X.shape[0]

        # Điều kiện dừng -> Tạo nút lá chứa giá trị trung bình cộng
        if depth >= self.max_depth or n_samples < self.min_samples_split or len(np.unique(y)) == 1:
            return {'value': np.mean(y)}

        best_feat, best_thresh = self._best_split(X, y)
        if best_feat is None:
            return {'value': np.mean(y)}

        left_mask = X[:, best_feat] <= best_thresh
        right_mask = ~left_mask

        # Đệ quy xây dựng cây con bên trái và bên phải
        left_child = self._build_tree(X[left_mask], y[left_mask], depth + 1)
        right_child = self._build_tree(X[right_mask], y[right_mask], depth + 1)

        return {
            'feature': best_feat,
            'threshold': best_thresh,
            'left': left_child,
            'right': right_child
        }

    def fit(self, X, y):
        self.root = self._build_tree(X, y)

    # Dự đoán 1 dòng dữ liệu bằng cách đi theo các nhánh cây
    def _predict_row(self, node, x):
        if 'value' in node:
            return node['value']
        if x[node['feature']] <= node['threshold']:
            return self._predict_row(node['left'], x)
        return self._predict_row(node['right'], x)

    def predict(self, X):
        return np.array([self._predict_row(self.root, x) for x in X])

# Tận dụng dữ liệu Target Encoded X_train_te và y_train_te từ Ô 4
cart_scratch = DecisionTreeScratch(max_depth=5, min_samples_split=20, min_samples_leaf=10)
cart_scratch.fit(X_train_te.values, y_train_te.values)

# Dự đoán và tính chỉ số
y_pred_log_scratch = cart_scratch.predict(X_test_te.values)
y_pred_orig_scratch = np.maximum(0, np.expm1(y_pred_log_scratch))

rmsle = np.sqrt(mean_squared_error(y_test_te, y_pred_log_scratch))
rmse = np.sqrt(mean_squared_error(y_test_orig, y_pred_orig_scratch))
mae = mean_absolute_error(y_test_orig, y_pred_orig_scratch)

print("=== 4. CART REGRESSOR (CODE CHAY) ===")
print(f"RMSLE : {rmsle:.4f}")
print(f"RMSE  : {rmse:.2f} khách")
print(f"MAE   : {mae:.2f} khách")

=== 4. CART REGRESSOR (CODE CHAY) ===
RMSLE : 0.7950
RMSE  : 18.19 khách
MAE   : 12.79 khách
